# RFP Parsing P4 - HWPX 125 Eval-Covered Corpus

Deduplicated eval 500문항의 실제 정답 문서 40개를 반드시 포함하고, 균형 filler 85개를 추가해 125개 P4 HWPX corpus를 생성합니다.

핵심 차이:

- 기존 250/690처럼 P3 metadata를 필수 입력으로 쓰지 않습니다.
- `data/eval/*.csv`의 `ground_truth_docs`를 정규화해 실제 원본 source file로 매칭합니다.
- 선택된 125개 문서를 개선된 P4 parser로 다시 파싱합니다.
- fact chunk는 검색 목적별로 분리되고, low-confidence fact는 embed 대상에서 제외됩니다.


In [1]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

CORPUS_LIMIT = 125

PROJECT_ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/DB_RAG_Codeit_Project'),
    Path.cwd(),
    Path.cwd().parent,
]

def find_project_root() -> Path:
    for candidate in PROJECT_ROOT_CANDIDATES:
        module_path = candidate / "src" / "parsing" / "rfp_p4_hwpx_corpus.py"
        if module_path.exists():
            return candidate.resolve()
    raise FileNotFoundError("PROJECT_ROOT를 찾지 못했습니다. repo root에서 실행하거나 후보 경로를 수정하세요.")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src" / "parsing"))

from rfp_p4_hwpx_corpus import write_p4_corpus

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CORPUS_LIMIT:", CORPUS_LIMIT)
print("logs: write_p4_corpus(verbose=True, progress_every=1) will print per-document progress")


PROJECT_ROOT: /Users/apple/Desktop/codeit/project_2nd/chatbot
CORPUS_LIMIT: 125
logs: write_p4_corpus(verbose=True, progress_every=1) will print per-document progress


## 1. Preflight

필수 입력을 확인합니다. 125 경로는 P3 metadata가 없어도 실행되어야 합니다.


In [2]:
required_paths = {
    "eval_dir": PROJECT_ROOT / "data" / "eval",
    "original_data_dir": PROJECT_ROOT / "data" / "original_data_list",
    "hwpx_dir": PROJECT_ROOT / "data" / "hwpx_664",
    "g2b_master": PROJECT_ROOT / "data" / "g2b_master_cleaned.csv",
}

for name, path in required_paths.items():
    print(name, path, "exists=", path.exists())

eval_csvs = sorted(required_paths["eval_dir"].glob("*.csv")) if required_paths["eval_dir"].exists() else []
original_files = []
if required_paths["original_data_dir"].exists():
    for ext in ("*.hwp", "*.hwpx", "*.pdf"):
        original_files.extend(required_paths["original_data_dir"].rglob(ext))

print("eval_csv_count:", len(eval_csvs))
print("original_file_count:", len(original_files))
print("P3 metadata required for 125:", False)


g2b_path = required_paths["g2b_master"]
if g2b_path.exists():
    g2b_preview = pd.read_csv(g2b_path)
    print("g2b_master_cleaned.csv rows:", len(g2b_preview))
    print("g2b columns:", list(g2b_preview.columns))
    print("date policy: use only 입찰마감일시 inside 게시일시(입찰마감일시); do not use 게시일시")
else:
    print("g2b_master_cleaned.csv not found; G2B merge will be skipped")


eval_dir /Users/apple/Desktop/codeit/project_2nd/chatbot/data/eval exists= True
original_data_dir /Users/apple/Desktop/codeit/project_2nd/chatbot/data/original_data_list exists= True
hwpx_dir /Users/apple/Desktop/codeit/project_2nd/chatbot/data/hwpx_664 exists= True
g2b_master /Users/apple/Desktop/codeit/project_2nd/chatbot/data/g2b_master_cleaned.csv exists= True
eval_csv_count: 25
original_file_count: 690
P3 metadata required for 125: False
g2b_master_cleaned.csv rows: 701
g2b columns: ['업무구분', '업무여부', '구분', '입찰공고번호', '공고명', '공고기관', '수요기관', '게시일시(입찰마감일시)']
date policy: use only 입찰마감일시 inside 게시일시(입찰마감일시); do not use 게시일시


## 2. Generate P4 HWPX 125 Corpus


In [3]:
result = write_p4_corpus(PROJECT_ROOT, limit=CORPUS_LIMIT, verbose=True, progress_every=1)
print("output_dir:", result["output_dir"])
print("manifest selection:")
print(json.dumps(result["manifest"].get("selection", {}), ensure_ascii=False, indent=2))
print("\n[v2 validation]")
print(json.dumps(result["report_v2"], ensure_ascii=False, indent=2))


[p4] project_root=/Users/apple/Desktop/codeit/project_2nd/chatbot
[p4] output_dir=/Users/apple/Desktop/codeit/project_2nd/chatbot/outputs/parsing_p4_hwpx_125
[p4] limit=125
[selection] documents=125 eval=40
[selection] rule=eval-covered-125: include every physical source document matched from data/eval ground_truth_docs, then add balanced signal-rich and hard-distractor fillers
[selection] hard_distractors=73
[inputs] hwpx_lookup=663
[inputs] g2b_records=701 date_policy=bid_deadline_only
[parse] 001/125 status=success format=pdf tables=0 g2b=no_confident_match fact_blocks=7 chunks_v2=338 elapsed=9.70s file=고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
[parse] 002/125 status=success format=hwpx tables=107 g2b=no_confident_match fact_blocks=7 chunks_v2=150 elapsed=0.29s file=(사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp
[parse] 003/125 status=success format=hwpx tables=68 g2b=ambiguous_active fact_blocks=6 chunks_v2=134 elapsed=0.29s f

## 3. Required Checks


In [4]:
report = result["report_v2"]
selection = result["manifest"]["selection"]

actual_eval_docs = selection.get("actual_eval_unique_gt_docs")
expected_plan_eval_docs = selection.get("expected_eval_physical_source_docs")
if actual_eval_docs != expected_plan_eval_docs:
    print(f"WARNING: plan expected {expected_plan_eval_docs} eval docs, but current eval CSV has {actual_eval_docs} unique docs.")

checks = {
    "document_count == 125": report["document_count"] == 125,
    "all current eval GT docs included": selection.get("eval_physical_source_docs_included") == actual_eval_docs,
    "additional_sampled_docs fills to 125": selection.get("additional_sampled_docs") == 125 - actual_eval_docs,
    "missing_eval_gt_docs == []": selection.get("missing_eval_gt_docs") == [],
    "parse_success_docs == 125": report["parse_success_docs"] == 125,
    "duplicate_doc_id_count == 0": report["duplicate_doc_id_count"] == 0,
    "duplicate_chunk_id_count == 0": report["duplicate_chunk_id_count"] == 0,
    "duplicate_source_store_id_count == 0": report["duplicate_source_store_id_count"] == 0,
    "missing_source_store_ref == 0": report["missing_source_store_ref"] == 0,
    "low_confidence_fact_embedded_count == 0": report.get("low_confidence_fact_embedded_count", 0) == 0,
}

for name, ok in checks.items():
    print(f"{name}: {ok}")

failed = [name for name, ok in checks.items() if not ok]
if failed:
    raise AssertionError("failed checks: " + " | ".join(failed))


document_count == 125: True
all current eval GT docs included: True
additional_sampled_docs fills to 125: True
missing_eval_gt_docs == []: True
parse_success_docs == 125: True
duplicate_doc_id_count == 0: True
duplicate_chunk_id_count == 0: True
duplicate_source_store_id_count == 0: True
missing_source_store_ref == 0: True
low_confidence_fact_embedded_count == 0: True


## 4. Quick Preview


In [5]:
summary_df = result["summary_df"]
display_cols = [
    "rank_index", "source_file", "source_format", "parser_status",
    "is_eval_ground_truth", "table_count", "fact_block_count",
    "final_notice_id", "notice_id_status", "final_bid_deadline", "bid_deadline_status",
    "g2b_match_status", "g2b_notice_id", "g2b_notice_kind", "g2b_bid_deadline",
    "embedded_table_block_count", "suppressed_table_block_count",
]
display(summary_df[display_cols].head(20))

print("fact_type_counts:")
print(json.dumps(result["report_v2"].get("fact_type_counts", {}), ensure_ascii=False, indent=2))
print("table_role_counts:")
print(json.dumps(result["report_v2"].get("table_role_counts", {}), ensure_ascii=False, indent=2))

print("g2b_match_status_counts:")
print(json.dumps(result["report_v2"].get("g2b_match_status_counts", {}), ensure_ascii=False, indent=2))
print("date_policy:", result["report_v2"].get("date_policy"))


    rank_index                                                                               source_file source_format parser_status  is_eval_ground_truth  table_count  fact_block_count   final_notice_id     notice_id_status final_bid_deadline bid_deadline_status    g2b_match_status      g2b_notice_id g2b_notice_kind  g2b_bid_deadline  embedded_table_block_count  suppressed_table_block_count
0            1                                고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf           pdf       success                  True            0                 7                                missing                                missing  no_confident_match                                                                                0                             0
1            2     (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp          hwpx       success                  True          107                 7                    rejec

In [6]:
table_preview_df = result["table_preview_df"]
if table_preview_df.empty:
    print("table preview is empty")
else:
    pd.set_option("display.max_colwidth", 240)
    display(table_preview_df.head(20))


                                                                              source_file section_path                                                                     table_shape                                                                                      columns_candidate        table_role  table_signal_score      table_embed_reason                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             